In [ ]:
from langchain_prompts import DATE_ASSISTANT_SYSTEM_PROMPT
from datetime import datetime
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages([
    ("system", DATE_ASSISTANT_SYSTEM_PROMPT),
    ("human", "{user_query}")
])

example_user_query = """
                        What is yesterday's date?
                     """

messages = prompt.format_messages(user_query = example_user_query, 
                                    todays_date = datetime.now().strftime("%Y-%m-%d"))  # Shows message objects ready for a chat model

messages




Invoke a nebius llm model

In [ ]:
import os
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model
from langchain_openai import ChatOpenAI

env_path = Path(r"C:\Meher\codebase\stockanalysis\.env")
load_dotenv(env_path)

llm = ChatOpenAI(
    model=os.environ["NEBIUS_MODEL"],
    api_key=os.environ["NEBIUS_API_KEY"],
    base_url="https://api.tokenfactory.nebius.com/v1/",
    temperature=0,
    use_responses_api=False,
)

response = llm.invoke(messages)  # Sends the messages to the chat model and returns the response
print(response.content)  # Prints the response from the chat model

List all the models available to use

In [ ]:
import os
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv(".env")

client = OpenAI(
    api_key=os.environ["NEBIUS_API_KEY"],
    base_url="https://api.tokenfactory.nebius.com/v1/",
)

print("Configured model:", repr(os.environ.get("NEBIUS_MODEL")))

for model in client.models.list().data:
    print(model.id)

Tool Calling

In [ ]:
from langchain.tools import tool
from langchain.messages import HumanMessage
import math

@tool
def circle_area(radius: float) -> float: 
    """Return the area of a circle for a given radius."""
    return math.pi * radius * radius

tools = [circle_area]

# Bind tools to the model so it can decide to call them
llm_with_tools = llm.bind_tools(tools)



msg = HumanMessage(content="I have a circular garden of radius 3. What's the area?")
#msg = HumanMessage(content="I have a triangular garden of length 5 and height 3. What's the area?")
ai_msg = llm_with_tools.invoke([msg])

print("Model output Type", type(ai_msg))
print("Model output", ai_msg)
print("Tool calls (if any):", getattr(ai_msg, "tool_calls", None))

In [ ]:
final = None
if getattr(ai_msg, "tool_calls", None):
    for tool_call in ai_msg.tool_calls:
        if tool_call["name"] == "circle_area":
            r = float(tool_call["args"]["radius"])
            result = circle_area.invoke({"radius": r})
            final = f"The area is approximately {result:.2f} square units."
else:
    final = ai_msg.content

print("Final:", final)



Create An Agent

In [ ]:
from langchain.agents import create_agent
from langchain.tools import tool
import math

@tool
def circle_area(radius: float) -> float:
    """Return the area of a circle for a given radius.

    Args:
        radius: The radius of the circle
    """
    return math.pi * radius * radius

@tool
def rectangle_area(length: float, width: float) -> float:
    """Return the area of a rectangle given its length and width.

    Args:
        length: The length of the rectangle
        width: The width of the rectangle
    """
    return length * width

agent = create_agent(
    model=llm,
    tools=[circle_area, rectangle_area],
    system_prompt="You are a helpful geometry assistant. Always show your calculations."
)

In [ ]:
question = "I have a circular pond with radius 3 and a rectangular patio that is 4 by 5. What is the combined area of both?"

result = agent.invoke({"messages": [{"role": "user", "content": question}]})

print("Final Answer:", result)

# Peek under the hood — the agent called both tools automatically
print("\n--- Agent's steps ---")
for msg in result["messages"]:
    label = type(msg).__name__
    if hasattr(msg, "tool_calls") and msg.tool_calls:
        print(f"{label}: called → {[tc['name'] for tc in msg.tool_calls]}")
    elif msg.content:
        print(f"{label}: {str(msg.content)[:80]}")

In [ ]:
## Simple Q/A Chatbot
from langgraph.checkpoint.memory import InMemorySaver
from langchain.tools import tool
from langchain.agents import create_agent
import math

@tool
def circle_area(radius: float) -> float:
    """Return the area of a circle for a given radius.

    Args:
        radius: The radius of the circle
    """
    return math.pi * radius * radius

@tool
def rectangle_area(length: float, width: float) -> float:
    """Return the area of a rectangle given its length and width.

    Args:
        length: The length of the rectangle
        width: The width of the rectangle
    """
    return length * width

agent = create_agent(
    model=llm,
    tools=[circle_area, rectangle_area],
    system_prompt="You are a helpful geometry assistant. Always show your calculations.",
    checkpointer=InMemorySaver()
)

config = {"configurable": {"thread_id": "geometry-chat"}}

print("Geometry Chatbot ready! Type 'quit' to exit.\n")

while True:
    user_input = input("You: ").strip()
    if not user_input:
        continue
    if user_input.lower() in ("quit", "exit"):
        print("Goodbye!")
        break

    result = agent.invoke(
        {"messages": [{"role": "user", "content": user_input}]},
        config=config
    )
    print(f"Agent: {result['messages'][-1].content}\n")